# Copy Workflow Configuration #

This sample will create a new Workflow item and populate it with configuration data from an existing Workflow item. This data includes the system settings, diagram, job templates and roles.

### Make connections ###

In [19]:
import arcgis
from arcgis.gis.workflowmanager import WorkflowManager, WorkflowManagerAdmin
gis = arcgis.gis.GIS(profile="ps0008227", verify_cert=False)
dest_gis = arcgis.gis.GIS(profile="ps0008536", verify_cert=False)
source_item = gis.content.search('title:"Python Sample"')[0]
new_item_id = WorkflowManagerAdmin(dest_gis).create_item('Duplicated Workflow Sample')
print(f'Created item {new_item_id}')
dest_item = dest_gis.content.get(new_item_id)
source = WorkflowManager(source_item)
dest = WorkflowManager(dest_item)
# Could also update sharing properties here

Created item 8962a927964244ccb5507a9e9753400a


### Copy Data ###

In [20]:
# Copy Settings
# Note that the password won't be copied
dest.update_settings(source.settings)

True

In [21]:
# Copy Diagrams
existing_diagrams = [x.diagram_id for x in dest.diagrams]
diagram_ids = [x.diagram_id for x in source.diagrams if not(x.diagram_id in existing_diagrams)]
diagram_id_map = {}
for id in diagram_ids:
    d = source.diagram(id)
    new_id = dest.create_diagram(d.diagram_name, d.steps, d.display_grid, d.description, d.diagram_version >= 0, d.annotations, 
                                 d.data_sources) # Remove datasources here if you don't want to copy
    diagram_id_map.update({id: new_id})
    print(f'Created diagram {d.diagram_name}. ID = {new_id}')
diagram_id_map

Created diagram Fix Damaged infrastructure. ID = OiawotDsSA6cuveKvwxxIw


{'imzc5iluTWeIB3HNnplf9Q': 'OiawotDsSA6cuveKvwxxIw'}

In [22]:
# Copy Job Templates
existing_job_templates = [x.job_template_id for x in dest.job_templates]
job_template_ids = [x.job_template_id for x in source.job_templates if not(x.job_template_id in existing_job_templates)]
for id in job_template_ids:
    t = source.job_template(id)
    new_diagram_id = diagram_id_map[t.diagram_id]
    result = dest.create_job_template(t.job_template_name, t.default_priority_name, t.job_template_id, t.category, 
                                      t.default_job_duration, t.default_assigned_to, t.default_due_date, 
                                      t.default_start_date, t.job_start_date_type, new_diagram_id, t.diagram_name,
                                      t.default_assigned_type, t.description, t.default_description, 
                                      t.state, t.last_updated_by, t.last_updated_date, t.extended_property_table_definitions)
    print(f'Created job template {t.job_template_name}. ID = {result}')
    

WrkdpVMVTdWNSeSgE2w1rw
Created job template Fix Damaged infrastructure. ID = KC-Y6z_YQxuSXy1UzxRw8A


In [25]:
# Roles
dest_roles = [x.role_name for x in dest.wm_roles]
new_roles = [x.role_name for x in source.wm_roles if not(x.role_name in dest_roles)]
print(f'Creating roles {new_roles}')
for name in new_roles:
    r = source.wm_role(name)
    dest.create_wm_role(name, r.description, r.privileges)

Creating roles ['testRole']
